# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a scoring / ranking task, not classification in the pure sense — the end product is a priority queue for editors (top-N pages to review this week), not a single yes/no verdict on any one page. In practice I build it via a classifier (random forest predicting is_declining_label) and use its predicted probability as the score to rank pages — a common pattern where a classification model's output powers a ranking. Per framing-ml-problems: "Which ones first?" maps to ranking/scoring with a priority score as the target and precision@K as the metric — that's exactly this shape.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining_label, defined as trend_direction == "down". Per the label-trap rule in flyrank-data, trend_direction and trend_pct are never features — they define the label itself.

Is this observed or defined-by-rule? It's genuinely observed: trend_direction/trend_pct are computed from real measured traffic in two actual time windows (impressions_last_30d vs impressions_prev_30d, etc.), not from an arbitrary hand-picked threshold like "if impressions > 500." So the underlying trend is a real fact about the page's history, not someone's opinion of what "declining" should mean.

That said, it's a proxy, not a perfect target: a page can show trend_direction == down for reasons that have nothing to do with content quality — seasonality, a SERP layout change, a competitor's new page — so "flagged as declining" is a proxy for "worth an editor's look," not a certified diagnosis of a content problem.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50: of the top 50 pages my score ranks highest, what fraction are actually labeled declining? This is the right metric because the editor only ever looks at the top of the queue — recall across all 30,000 pages doesn't matter if the top 50 are wrong. A baseline hand-written rule (stale AND high-impressions) scores 0.240 Precision@50 on this data; that's the number a fixed rule has to beat.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one pseudonymized content item (a single page) for one client, with trailing-90-day performance metrics. Loaded below.

In [4]:
import os

if not os.path.exists("flyrank-ml-track"):
    !git clone https://github.com/muneeb-khokhar/flyrank-ml-track.git

os.chdir("flyrank-ml-track")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

cols = ["content_id", "client_id", "impressions_90d", "sessions_90d", "avg_position",
        "ctr", "days_since_last_update", "word_count", "trend_direction"]

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print(f"One row = one content item for one client. Sample:")
df[cols].sample(5, random_state=7)

Cloning into 'flyrank-ml-track'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 38), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 5.53 MiB/s, done.
Resolving deltas: 100% (38/38), done.
Rows: 30,000  |  Columns: 44
One row = one content item for one client. Sample:


,content_id,client_id,impressions_90d,sessions_90d,avg_position,ctr,days_since_last_update,word_count,trend_direction
1252,content_a8c35eeef547,client_19581e27de,82428,370,3.9,0.41,22,NaN,stable
10444,content_26eb8cf5167c,client_4fc82b26ae,864,11,8.5,0.12,20,1553.0,down
8994,content_663ee9569ab2,client_349c41201b,3544,11,7.1,0.34,20,2613.0,stable
7463,content_7e13c4bf098b,client_f369cb89fc,2,2,8.0,0.00,8,2698.0,flat
1910,content_b9b75b2367a3,client_bdd2d3af3a,118,2,8.9,0.85,8,2663.0,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "stale (>180 days since update) AND high impressions (≥500)" is transparent but rigid — it can only combine signals with hard thresholds and AND/OR logic. My own pipeline run shows this rule scores 0.240 Precision@50. A random forest trained on the same signals (impressions, CTR, position, staleness, word count) scores 0.740 Precision@50 — about 3.1x better — because it can weigh many weak, correlated signals together and find nonlinear combinations (e.g. "moderately stale AND mid-position AND low CTR" matters more than any one signal alone) that no single if-statement captures.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.